In [1]:
print("it is starting")
import pandas as pd
import os 
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

it is starting


In [2]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [3]:
seed_value = 42
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)  # For multi-GPU.
torch.backends.cudnn.deterministic = True  # Ensure deterministic behavior
torch.backends.cudnn.benchmark = False

In [4]:
train_folder = "/kaggle/input/mamba-dataset/train"  # Replace with the full path to your train folder
test_folder = "/kaggle/input/mamba-dataset/test"

In [5]:
def create_sequences(data, lag=10):
    X, y = [], []
    for i in range(lag, len(data) - 1):  # Loop starting from lag to the second last element
        # Extract position (x) and force (f) sequences within the given lag range
        position_sequence = data[i - lag: i, 1]  # Take position from lag to lag+1
        force_sequence = data[i - lag: i + 1, 0]  # Force sequence from lag to lag+1 (1 more than position)ac

        # Ensure the shape of X has the correct number of features (lag * 2)
        X.append(np.concatenate((force_sequence, position_sequence), axis=0))

        # Target is the next position (displacement) at time (i+1)
        y.append(data[i, 1])  

    return np.array(X), np.array(y)

In [6]:
def load_and_combine_csv_files(train_folder, test_folder, num_train_files=50, num_test_files=20):
    # Initialize an empty list to store dataframes
    all_data = []

    print("Training files are:")
    # Load CSV files from the train folder, with filenames from data_1_train.csv to data_50_train.csv
    for i in range(1, num_train_files + 1):  # File numbers from 1 to 50
        file_name = f"/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_{i}_train.csv"  # Construct the file name
        file_path = os.path.join(train_folder, file_name)
        if os.path.exists(file_path):  # Check if the file exists
            print(file_path)
            data = pd.read_csv(file_path)
            data = data.drop(index=0)  # Drop the first row (if needed)
            all_data.append(data)  # Append the dataframe to the list
        else:
            print(f"File {file_name} not found in the folder.")

    print("Testing files are:")
    # Load CSV files from the test folder, with filenames from data_1_test.csv to data_20_test.csv
    for i in range(1, num_test_files + 1):  # File numbers from 1 to 20
        file_name = f"/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/test/data_{i}_test.csv"  # Construct the file name
        file_path = os.path.join(test_folder, file_name)
        if os.path.exists(file_path):  # Check if the file exists
            print(file_path)
            data = pd.read_csv(file_path)
            data = data.drop(index=0)  # Drop the first row (if needed)
            all_data.append(data)  # Append the dataframe to the list
        else:
            print(f"File {file_name} not found in the folder.")

    # Combine all dataframes row-wise
    combined_data = pd.concat(all_data, ignore_index=True)
    
    return combined_data

In [7]:
combined_data = load_and_combine_csv_files(train_folder, test_folder)
print(combined_data)

Training files are:
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_1_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_2_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_3_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_4_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_5_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_6_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_7_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_8_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_9_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_10_train.csv
/kaggle/input/datasets/rajinikanthadithya/bosch-internship-dataset/train/data_1

In [8]:
scaler_f = MinMaxScaler()
scaler_x = MinMaxScaler()


combined_data[['f']] = scaler_f.fit_transform(combined_data[['f']])

# Scale the 'x' column using scaler_x
combined_data[['x']] = scaler_x.fit_transform(combined_data[['x']])


combined_data = combined_data[['f', 'x']]


In [9]:
def split_data_into_parts(combined_data, num_train_parts=50, num_test_parts=20, train_size=2500, test_size=10000):
    # List to hold the parts
    data_parts = []

    # Split the data into the first 50 parts (each of size 2500 rows)
    for i in range(num_train_parts):
        start_idx = i * train_size
        end_idx = (i + 1) * train_size if i < num_train_parts - 1 else num_train_parts * train_size  # Handle the last part
        data_parts.append(combined_data[start_idx:end_idx])

    # Split the data into the next 20 parts (each of size 10000 rows)
    for i in range(num_test_parts):
        start_idx = num_train_parts * train_size + i * test_size
        end_idx = (num_train_parts * train_size + (i + 1) * test_size) if i < num_test_parts - 1 else len(combined_data)
        data_parts.append(combined_data[start_idx:end_idx])

    return data_parts

In [10]:
combined_data = np.array(combined_data)  # Assuming combined_data is a numpy array

# Split the data into 70 parts, with the first 50 having 2500 rows each and the last 20 having 10000 rows each
data_parts = split_data_into_parts(combined_data, num_train_parts=50, num_test_parts=20, train_size=2500, test_size=10000)


# In[10]:


train_parts = data_parts[:50]  # First 50 parts for training
test_parts = data_parts[50:]   # Last 20 parts for testing

# You can now use `train_parts` and `test_parts` for further processing or model training.
print(train_parts[0].shape)
print(test_parts[0].shape)

lag = 10

X_train_list, y_train_list = [], []
for part in train_parts:
    X, y = create_sequences(part, lag)
    X_train_list.append(X)
    y_train_list.append(y)

X_test_list, y_test_list = [], []
for part in test_parts:
    X, y = create_sequences(part, lag)
    X_test_list.append(X)
    y_test_list.append(y)

X_train = np.vstack(X_train_list)

X_test = np.vstack(X_test_list)

y_train = []
for j in y_train_list:
    for i in j:
        y_train.append(i)

y_train = np.array(y_train)

y_test = []
for j in y_test_list:
    for i in j:
        y_test.append(i)
        
y_test = np.array(y_test)

input_dim = 2*lag+1


print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

(2500, 2)
(10000, 2)
X_train shape: (124450, 21)
y_train shape: (124450,)
X_test shape: (199780, 21)
y_test shape: (199780,)


In [11]:
def recursive_prediction_for_file(model, input_seq, force_values, lag, device, n_predictions=10000):
    predictions = []

    # Ensure input_seq is a 1-D tensor
    if isinstance(input_seq, np.ndarray):
        input_seq = torch.tensor(input_seq, dtype=torch.float32).to(device)
        
    input_seq = input_seq.float()  # Convert to tensor if it's an ndarray
   

    for i in range(n_predictions):  
        with torch.no_grad():
            input_seq = input_seq.to(device)
            input_seq = input_seq.view(1, -1)  # Reshape for batch dimension
            pred = model(input_seq)  # Add batch dimension for the prediction
            predictions.append(pred.cpu().numpy().flatten())  # Flatten the prediction and append it to the list
            
            # Get the predicted displacement and the actual force for the next timestep
            predicted_displacement = pred.cpu().numpy().flatten()[0]  # Flatten the predicted displacement to scalar
            actual_force = force_values[i + lag+1]
            
            input_seq = modify_tensor(input_seq, lag, actual_force, predicted_displacement)
           
    return np.array(predictions)

In [12]:
def modify_tensor(tensor, lag, x, y):
  # Get the device of the input tensor (e.g., cuda:0 or cpu)
  device = tensor.device

  tensor = tensor.squeeze()  # Converts (1, 21) to (21,) if necessary

  # Remove the 1st element and the element at (lag+1)th index
  tensor_without_first_and_lagplus1 = torch.cat(
      (tensor[1 : lag + 1], tensor[lag + 2 :])
  )  # Remove 1st and lag+1th

  # Add y at the end of the tensor (explicitly on the same device)
  y_tensor = torch.tensor([y], dtype=torch.float32, device=device)
  tensor_with_y_at_end = torch.cat(
      (tensor_without_first_and_lagplus1, y_tensor)
  )

  # Add x at the lagth index (explicitly on the same device)
  x_tensor = torch.tensor([x], dtype=torch.float32, device=device)
  modified_tensor = torch.cat(
      (tensor_with_y_at_end[:lag], x_tensor, tensor_with_y_at_end[lag:])
  )

  return modified_tensor.unsqueeze(0)

def make_predictions(model, X_test, force_values, device, lag):
    prediction_full = []
    m = int(len(X_test) // 20)  # Number of recursive prediction steps for each file
    m_force = int(len(force_values)//20)
    print("value of m is ",m)
    
    for i in range(20):  # Process each test file
        # Get input sequence and corresponding force values for each file
        input_seq = X_test[i * m]  # Test input data
        force_vals = test_parts[i][:,0] # Corresponding force values
        
        predictions = recursive_prediction_for_file(model, input_seq, force_vals, lag, device, m)
        prediction_full.append(predictions)
    
    # Flatten the list of predictions
    prediction_final = [j for i in prediction_full for j in i]
    prediction_final = np.array(prediction_final)
    
    return prediction_final

In [13]:
class FNO1d(nn.Module):

  def __init__(self, input_dim, hidden_dim=64, modes=16):
    super(FNO1d, self).__init__()
    self.input_dim = input_dim
    self.hidden_dim = hidden_dim
    self.modes = modes

    # Lifting layer
    self.fc0 = nn.Linear(input_dim, hidden_dim)

    # Fourier spectral weight parameters
    self.weights = nn.Parameter(
        torch.randn(hidden_dim, hidden_dim, modes, dtype=torch.cfloat)
        * (1 / hidden_dim)
    )

    # Projection layers
    self.fc1 = nn.Linear(hidden_dim, 128)
    self.fc2 = nn.Linear(128, 1)

  def compl_mul1d(self, input, weights):
    # (batch, in_channel, x) * (in_channel, out_channel, x) -> (batch, out_channel, x)
    return torch.einsum("bix,iox->box", input, weights)

  def forward(self, x):
    if isinstance(x, np.ndarray):
      x = torch.tensor(x, dtype=torch.float32)

    # Lifting
    x = self.fc0(x)  # (batch, input_dim) -> (batch, hidden_dim)
    x = x.unsqueeze(2)  # (batch, hidden_dim, 1) to mimic spatial/temporal dim

    batchsize = x.shape[0]
    x_ft = torch.fft.rfft(x)

    # Multiply relevant Fourier modes
    out_ft = torch.zeros(
        batchsize,
        self.hidden_dim,
        x.size(-1) // 2 + 1,
        device=x.device,
        dtype=torch.cfloat,
    )
    modes_to_use = min(self.modes, x_ft.size(-1))
    out_ft[:, :, :modes_to_use] = self.compl_mul1d(
        x_ft[:, :, :modes_to_use], self.weights[:, :, :modes_to_use]
    )

    # Return to physical space
    x = torch.fft.irfft(out_ft, n=x.size(-1))

    x = x.squeeze(2)
    x = torch.relu(self.fc1(x))
    x = self.fc2(x)
    return x


def trainFNO(
    X_train, y_train, epochs=1000, batch_size=128, lr=0.001, patience=10000
):
  X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

  train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

  model = FNO1d(input_dim=X_train.shape[1], hidden_dim=64, modes=16)
  print(model)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  print("model connected to", device)

  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=lr)

  best_train_loss = float("inf")
  epochs_without_improvement = 0

  for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for data, target in train_loader:
      data, target = data.to(device), target.to(device)
      optimizer.zero_grad()
      outputs = model(data)
      loss = criterion(outputs.squeeze(), target.squeeze())
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    if epoch_loss < best_train_loss:
      best_train_loss = epoch_loss
      epochs_without_improvement = 0
    else:
      epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
      break
    if epoch % 10 == 0:
      print(f"Epoch {epoch+1}/{epochs}, Average Train Loss: {epoch_loss:.8f}")

  return model
    

In [19]:
class LNO1d(nn.Module):

  def __init__(self, input_dim, hidden_dim=64, modes=16):
    super(LNO1d, self).__init__()
    self.input_dim = input_dim
    self.hidden_dim = hidden_dim
    self.modes = modes

    self.fc0 = nn.Linear(input_dim, hidden_dim)

    # Weights configured to transform hidden_dim -> hidden_dim across modes
    self.weights_real = nn.Parameter(
        torch.randn(hidden_dim, hidden_dim, modes) * (1 / hidden_dim)
    )
    self.weights_imag = nn.Parameter(
        torch.randn(hidden_dim, hidden_dim, modes) * (1 / hidden_dim)
    )

    # After processing, the feature dimension matches hidden_dim
    self.fc1 = nn.Linear(hidden_dim, 128)
    self.fc2 = nn.Linear(128, 1)

  def forward(self, x):
    if isinstance(x, np.ndarray):
      x = torch.tensor(x, dtype=torch.float32)

    # 1. Lifting: (batch, input_dim) -> (batch, hidden_dim)
    x = self.fc0(x)

    # Add a pseudo-spatial/mode dimension for transform: (batch, hidden_dim, 1)
    x = x.unsqueeze(2)
    batch_size = x.shape[0]

    # Expand/simulate transform across the configured modes
    x_complex = torch.complex(x, torch.zeros_like(x))
    # Expand to match modes dimension: (batch, hidden_dim, modes)
    x_complex = x_complex.expand(-1, -1, self.modes)

    weights = torch.complex(self.weights_real, self.weights_imag)

    # Batched matrix multiplication across channels: (batch, hidden_dim, modes)
    out_complex = torch.einsum("bix,iox->box", x_complex, weights)

    # Reduce across modes to return to (batch, hidden_dim)
    x_out = out_complex.mean(dim=2).real

    # 2. Fully connected projection layers
    x_out = torch.relu(self.fc1(x_out))
    return self.fc2(x_out)


def trainLNO(
    X_train, y_train, epochs=1000, batch_size=128, lr=0.001, patience=10000
):
  X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

  train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

  model = LNO1d(input_dim=X_train.shape[1], hidden_dim=64, modes=16)
  print(model)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  print("model connected to", device)

  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=lr)

  best_train_loss = float("inf")
  epochs_without_improvement = 0

  for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for data, target in train_loader:
      data, target = data.to(device), target.to(device)
      optimizer.zero_grad()
      outputs = model(data)
      loss = criterion(outputs.squeeze(), target.squeeze())
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    if epoch_loss < best_train_loss:
      best_train_loss = epoch_loss
      epochs_without_improvement = 0
    else:
      epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
      break
    if epoch % 10 == 0:
      print(f"Epoch {epoch+1}/{epochs}, Average Train Loss: {epoch_loss:.8f}")

  return model

In [15]:
class DeepONet(nn.Module):

  def __init__(self, branch_input_dim, trunk_input_dim=1, hidden_dim=128):
    super(DeepONet, self).__init__()

    # Branch net (processes input features / history window)
    self.branch = nn.Sequential(
        nn.Linear(branch_input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
    )

    # Trunk net (processes coordinates or query locations, default dim=1)
    self.trunk = nn.Sequential(
        nn.Linear(trunk_input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
    )

    self.bias = nn.Parameter(torch.zeros(1))

  def forward(self, x):
    if isinstance(x, np.ndarray):
      x = torch.tensor(x, dtype=torch.float32)

    # Split input into branch features and a dummy/default trunk coordinate if uniform
    # Assuming x contains all branch features, we can supply a default trunk coordinate vector of ones.
    branch_out = self.branch(x)

    # Default dummy trunk coordinates (e.g., evaluating at endpoint query coordinate 1.0)
    trunk_coord = torch.ones(x.size(0), 1, device=x.device)
    trunk_out = self.trunk(trunk_coord)

    # Dot product combination between branch and trunk networks + bias
    prediction = torch.sum(branch_out * trunk_out, dim=1, keepdim=True) + self.bias
    return prediction


def trainDeepONet(
    X_train, y_train, epochs=1000, batch_size=128, lr=0.001, patience=10000
):
  X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

  train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

  model = DeepONet(branch_input_dim=X_train.shape[1], hidden_dim=128)
  print(model)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  print("model connected to", device)

  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=lr)

  best_train_loss = float("inf")
  epochs_without_improvement = 0

  for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for data, target in train_loader:
      data, target = data.to(device), target.to(device)
      optimizer.zero_grad()
      outputs = model(data)
      loss = criterion(outputs.squeeze(), target.squeeze())
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    if epoch_loss < best_train_loss:
      best_train_loss = epoch_loss
      epochs_without_improvement = 0
    else:
      epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
      break
    if epoch % 10 == 0:
      print(f"Epoch {epoch+1}/{epochs}, Average Train Loss: {epoch_loss:.8f}")

  return model

In [16]:
def evaluate_and_plot_model(
    model_name,model, X_test, y_test, test_parts, device, lag, scaler_x, num_parts=20
):
  """Evaluates the model on test data, calculates metrics per part,

  and plots Actual vs Predicted for each part.
  """
  # 1. Make predictions
  preds = make_predictions(model, X_test, test_parts, device, lag)

  # 2. Inverse transform predictions and actuals
  preds = scaler_x.inverse_transform(preds.reshape(-1, 1)).flatten()
  y_test = scaler_x.inverse_transform(y_test.reshape(-1, 1)).flatten()

  # 3. Helper function for part-wise metrics inside scope
  def compute_metrics_in_parts(y_true, y_pred, parts):
    part_size = len(y_true) // parts
    mse_list, r2_list = [], []

    for i in range(parts):
      start_idx = i * part_size
      end_idx = len(y_true) if i == parts - 1 else (i + 1) * part_size

      true_part = y_true[start_idx:end_idx]
      pred_part = y_pred[start_idx:end_idx]

      mse_list.append(mean_squared_error(true_part, pred_part))
      r2_list.append(r2_score(true_part, pred_part))

    return mse_list, r2_list

  # Compute metrics across parts
  mse_list, r2_list = compute_metrics_in_parts(y_test, preds, num_parts)

  # Print simple MSE across 4 parts as in your snippet
  print("--- Part-wise MSE (4 parts) ---")
  part_size_4 = len(y_test) // 4
  for i in range(4):
    start_idx = i * part_size_4
    end_idx = len(y_test) if i == 3 else (i + 1) * part_size_4
    mse_part = mean_squared_error(
        y_test[start_idx:end_idx], preds[start_idx:end_idx]
    )
    print(f"MSE for Part {i+1}: {mse_part}")
  print("-" * 40)

  # 4. Plotting actual vs predicted for each part
  '''
  part_size_20 = len(y_test) // num_parts
  time_step = 0.1  # Time step in seconds
  time_array = np.arange(0, 1000, time_step)

  for i in range(num_parts):
    plt.figure(figsize=(20, 10))
    start_idx = i * part_size_20
    end_idx = (i + 1) * part_size_20

    file_prediction = preds[start_idx:end_idx]
    file_actual = y_test[start_idx:end_idx]

    plt.plot(
        time_array[0 : len(file_actual)],
        file_actual,
        label="Actual",
        color="blue",
        linestyle="-",
        linewidth=2,
    )
    plt.plot(
        time_array[0 : len(file_actual)],
        file_prediction,
        label="Predicted",
        color="red",
        linestyle="-",
        linewidth=2,
    )

    print(f"file {i+1}")
    print(f"mse is {mse_list[i]}")
    print(f"R2 score is {r2_list[i]}")
    print("-" * 18)

    plt.xlabel("Time (seconds)", fontsize=30)
    plt.ylabel("Value", fontsize=30)
    plt.title(
        f"Actual vs Predicted (File Number: {i + 1}) MSE: {mse_list[i]:.4f}",
        fontsize=30,
    )
    plt.xticks(fontsize=30)
    plt.yticks(fontsize=30)
    plt.legend(fontsize=30)
    plt.grid()

    plt.savefig(f"{model_name}_file_{i + 1}.png")
    plt.xlim(0, 1000)
    plt.show()
    '''

In [17]:
lr = 1e-4
epoch = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example for FNO
model_fno = trainFNO(X_train, y_train, epochs=epoch, lr=lr)
evaluate_and_plot_model(
    "FNO",
    model=model_fno,
    X_test=X_test,
    y_test=y_test,
    test_parts=test_parts,
    device=device,
    lag=lag,
    scaler_x=scaler_x,
    num_parts=20,
)

FNO1d(
  (fc0): Linear(in_features=21, out_features=64, bias=True)
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)
model connected to cuda
Epoch 1/1000, Average Train Loss: 0.01126076
Epoch 11/1000, Average Train Loss: 0.00000116
Epoch 21/1000, Average Train Loss: 0.00000042
Epoch 31/1000, Average Train Loss: 0.00000025
Epoch 41/1000, Average Train Loss: 0.00000021
Epoch 51/1000, Average Train Loss: 0.00000021
Epoch 61/1000, Average Train Loss: 0.00000019
Epoch 71/1000, Average Train Loss: 0.00000019
Epoch 81/1000, Average Train Loss: 0.00000018
Epoch 91/1000, Average Train Loss: 0.00000016
Epoch 101/1000, Average Train Loss: 0.00000016
Epoch 111/1000, Average Train Loss: 0.00000015
Epoch 121/1000, Average Train Loss: 0.00000015
Epoch 131/1000, Average Train Loss: 0.00000015
Epoch 141/1000, Average Train Loss: 0.00000013
Epoch 151/1000, Average Train Loss: 0.00000014
Epoch 161/1000, Average Train Loss: 0.0000001

In [20]:
lr = 1e-4
epoch = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example for FNO
model_fno = trainLNO(X_train, y_train, epochs=epoch, lr=lr)
evaluate_and_plot_model(
    "LNO",
    model=model_fno,
    X_test=X_test,
    y_test=y_test,
    test_parts=test_parts,
    device=device,
    lag=lag,
    scaler_x=scaler_x,
    num_parts=20,
)

LNO1d(
  (fc0): Linear(in_features=21, out_features=64, bias=True)
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)
model connected to cuda
Epoch 1/1000, Average Train Loss: 0.00998962
Epoch 11/1000, Average Train Loss: 0.00000088
Epoch 21/1000, Average Train Loss: 0.00000037
Epoch 31/1000, Average Train Loss: 0.00000029
Epoch 41/1000, Average Train Loss: 0.00000029
Epoch 51/1000, Average Train Loss: 0.00000028
Epoch 61/1000, Average Train Loss: 0.00000025
Epoch 71/1000, Average Train Loss: 0.00000024
Epoch 81/1000, Average Train Loss: 0.00000022
Epoch 91/1000, Average Train Loss: 0.00000022
Epoch 101/1000, Average Train Loss: 0.00000020
Epoch 111/1000, Average Train Loss: 0.00000018
Epoch 121/1000, Average Train Loss: 0.00000017
Epoch 131/1000, Average Train Loss: 0.00000016
Epoch 141/1000, Average Train Loss: 0.00000015
Epoch 151/1000, Average Train Loss: 0.00000013
Epoch 161/1000, Average Train Loss: 0.0000001

In [21]:
lr = 1e-4
epoch = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example for FNO
model_fno = trainDeepONet(X_train, y_train, epochs=epoch, lr=lr)
evaluate_and_plot_model(
    "DeepOnet",
    model=model_fno,
    X_test=X_test,
    y_test=y_test,
    test_parts=test_parts,
    device=device,
    lag=lag,
    scaler_x=scaler_x,
    num_parts=20,
)

DeepONet(
  (branch): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
  )
  (trunk): Sequential(
    (0): Linear(in_features=1, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
  )
)
model connected to cuda
Epoch 1/1000, Average Train Loss: 0.00141499
Epoch 11/1000, Average Train Loss: 0.00000169
Epoch 21/1000, Average Train Loss: 0.00000130
Epoch 31/1000, Average Train Loss: 0.00000102
Epoch 41/1000, Average Train Loss: 0.00000085
Epoch 51/1000, Average Train Loss: 0.00000074
Epoch 61/1000, Average Train Loss: 0.00000074
Epoch 71/1000, Average Train Loss: 0.00000058
Epoch 81/1000, Average Train Loss: 0.00000057
Epoch 91/1000, Average Train Loss: 0.00000048
Epoch 101/1000, A